# AI-Task1: LLM Showdown

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jaweriamushtaq41-coder/Smart_District_2/blob/main/AI-Task1/AI_Task1_LLM_Showdown.ipynb)

**Kya banana hai:** Ek prompt OpenAI, Claude, Gemini aur Groq ko bhejna aur muqabla dikhana: provider ka naam, jawab, kitni der lagi, kitne tokens aur andaza kitna kharcha. Keys `.env` mein.

**Concepts:** OpenAI / Claude / Gemini / Groq API, tokens, cost, `.env`

**Colab mein kaise chalayen:**
1. Upar *Open In Colab* dabao (ya ye file Colab mein upload karo).
2. *Runtime > Run all* dabao.
3. Keys ho to Colab ke left side ke **Secrets** (chaabi ka icon) mein daalo. Bina key ke bhi chalta hai (**demo mode**, nakli jawab).

> Notebook ko share karne se pehle apni keys kabhi output / cell mein mat chhodo.

## 1. Libraries install aur keys (`.env`)

In [1]:
!pip -q install -U python-dotenv openai anthropic google-generativeai groq

In [2]:
import os

# .env file banao. Keys do tareeqon se aa sakti hain:
#  1) Colab "Secrets" (left side ka chaabi wala icon), secret ka naam wahi jo neeche list mein hai, "Notebook access" ON
#  2) apni .env file Colab ke "Files" panel (left side ka folder icon) mein drag-drop kar do, run karne se pehle
NAMES = ["OPENAI_API_KEY", "ANTHROPIC_API_KEY", "GOOGLE_API_KEY", "GROQ_API_KEY"]

existing = {}                       # pehle se .env ho (upload ki hui) to uski keys bacha lo
if os.path.exists(".env"):
    with open(".env") as f:
        for line in f:
            if "=" in line and not line.startswith("#"):
                key, value = line.strip().split("=", 1)
                existing[key] = value

lines = []
for name in NAMES:
    value = existing.get(name, "")
    try:
        from google.colab import userdata
        value = userdata.get(name) or value
    except Exception:          # Colab nahi hai, ya ye secret nahi banaya
        pass
    lines.append(f"{name}={value}")

with open(".env", "w") as f:
    f.write("\n".join(lines) + "\n")

from dotenv import load_dotenv
load_dotenv(".env", override=True)

found = [n for n in NAMES if os.getenv(n)]
print("Keys mili:", found if found else "koi nahi (demo mode chalega)")

Keys mili: koi nahi (demo mode chalega)


## 2. Helper: chaaron providers ek jaisa
`ask(provider, prompt, ...)` har provider ke SDK ko bulata hai aur ek hi shakal ka `Reply` deta hai
(text, time, tokens, cost). `demo=True` par nakli jawab milta hai. Ye cell sirf chalana hai, samajhne ke liye parh lo.

In [3]:
import os
import time
from dataclasses import dataclass

from dotenv import load_dotenv

load_dotenv()   # .env file se keys parh leta hai

PROVIDERS = ["openai", "claude", "gemini", "groq"]

# .env mein model ka naam badal sakte ho (model ke naam jaldi purane ho jate hain)
DEFAULT_MODELS = {
    "openai": os.getenv("OPENAI_MODEL", "gpt-4o-mini"),
    "claude": os.getenv("ANTHROPIC_MODEL", "claude-3-5-haiku-latest"),
    "gemini": os.getenv("GEMINI_MODEL", "gemini-1.5-flash"),
    "groq": os.getenv("GROQ_MODEL", "llama-3.1-8b-instant"),
}

KEY_NAMES = {
    "openai": "OPENAI_API_KEY",
    "claude": "ANTHROPIC_API_KEY",
    "gemini": "GOOGLE_API_KEY",
    "groq": "GROQ_API_KEY",
}

# (input, output) dollar per 1 million tokens. Ye sirf ANDAZA hai, asli rate provider ki site par dekho.
PRICES = {
    "openai": (0.15, 0.60),
    "claude": (0.80, 4.00),
    "gemini": (0.075, 0.30),
    "groq": (0.05, 0.08),
}


# demo mein har provider ki nakli raftaar (Groq sab se tez, jaisa asal mein hota hai)
DEMO_DELAY = {"openai": 0.12, "claude": 0.18, "gemini": 0.09, "groq": 0.03}


class LLMError(Exception):
    """Provider se jawab na aaye ya key / SDK ka masla ho."""


@dataclass
class Reply:
    provider: str
    model: str
    text: str
    seconds: float
    input_tokens: int
    output_tokens: int
    cost_usd: float
    demo: bool = False

    @property
    def total_tokens(self):
        return self.input_tokens + self.output_tokens


def has_key(provider):
    return bool(os.getenv(KEY_NAMES[provider], "").strip())


def available_providers():
    """Jin providers ki key .env mein mojood hai."""
    return [p for p in PROVIDERS if has_key(p)]


def estimate_cost(provider, input_tokens, output_tokens):
    price_in, price_out = PRICES[provider]
    return (input_tokens * price_in + output_tokens * price_out) / 1_000_000


def rough_tokens(text):
    # angrezi mein taqreeban 4 huroof = 1 token (sirf andaza)
    return max(1, len(text) // 4)


# ---------------- har provider ka apna call ----------------
def _call_openai(model, prompt, system, temperature, max_tokens):
    from openai import OpenAI
    client = OpenAI(api_key=os.getenv(KEY_NAMES["openai"]))
    messages = []
    if system:
        messages.append({"role": "system", "content": system})
    messages.append({"role": "user", "content": prompt})
    r = client.chat.completions.create(
        model=model, messages=messages, temperature=temperature, max_tokens=max_tokens)
    return r.choices[0].message.content, r.usage.prompt_tokens, r.usage.completion_tokens


def _call_groq(model, prompt, system, temperature, max_tokens):
    from groq import Groq
    client = Groq(api_key=os.getenv(KEY_NAMES["groq"]))
    messages = []
    if system:
        messages.append({"role": "system", "content": system})
    messages.append({"role": "user", "content": prompt})
    r = client.chat.completions.create(
        model=model, messages=messages, temperature=temperature, max_tokens=max_tokens)
    return r.choices[0].message.content, r.usage.prompt_tokens, r.usage.completion_tokens


def _call_claude(model, prompt, system, temperature, max_tokens):
    import anthropic
    client = anthropic.Anthropic(api_key=os.getenv(KEY_NAMES["claude"]))
    kwargs = {"model": model, "max_tokens": max_tokens, "temperature": temperature,
              "messages": [{"role": "user", "content": prompt}]}
    if system:
        kwargs["system"] = system          # Claude mein system alag parameter hota hai
    r = client.messages.create(**kwargs)
    return r.content[0].text, r.usage.input_tokens, r.usage.output_tokens


def _call_gemini(model, prompt, system, temperature, max_tokens):
    import google.generativeai as genai
    genai.configure(api_key=os.getenv(KEY_NAMES["gemini"]))
    gm = genai.GenerativeModel(model, system_instruction=system)
    r = gm.generate_content(
        prompt, generation_config={"temperature": temperature, "max_output_tokens": max_tokens})
    usage = r.usage_metadata
    return r.text, usage.prompt_token_count, usage.candidates_token_count


CALLERS = {"openai": _call_openai, "claude": _call_claude, "gemini": _call_gemini, "groq": _call_groq}


def _demo_reply(provider, prompt, mock_reply):
    """Nakli jawab: mock_reply (text ya function) ho to wahi, warna ek chhota sa default."""
    if callable(mock_reply):
        text = mock_reply(prompt)
    elif mock_reply is not None:
        text = mock_reply
    else:
        text = (f"[DEMO] Ye {provider} ka nakli jawab hai. Asli jawab ke liye .env mein "
                f"{KEY_NAMES[provider]} likho.")
    return text


def ask(provider, prompt, system=None, temperature=0.7, max_tokens=500,
        demo=False, mock_reply=None, retries=2):
    """Ek provider ko prompt bhejo aur Reply wapas lo (text, time, tokens, cost)."""
    if provider not in CALLERS:
        raise LLMError(f"Unknown provider '{provider}'. Choose from {PROVIDERS}")
    model = DEFAULT_MODELS[provider]
    start = time.perf_counter()

    if demo:
        text = _demo_reply(provider, prompt, mock_reply)
        in_tok = rough_tokens((system or "") + prompt)
        out_tok = rough_tokens(text)
        time.sleep(DEMO_DELAY[provider])         # thora sa waqt lagne ka ehsaas
    else:
        if not has_key(provider):
            raise LLMError(f"{KEY_NAMES[provider]} .env mein nahi mili")
        for attempt in range(retries + 1):
            try:
                text, in_tok, out_tok = CALLERS[provider](model, prompt, system, temperature, max_tokens)
                break
            except ImportError as e:
                raise LLMError(f"{provider} ka SDK install nahi hai: {e}") from e
            except Exception as e:                # SDK ke alag alag errors ek jaisa kar diye
                too_many = "429" in str(e) or "rate" in str(e).lower()
                if too_many and attempt < retries:
                    time.sleep(2 ** attempt)      # 1s, 2s ruk kar dobara
                    continue
                raise LLMError(f"{provider} error: {e}") from e

    seconds = time.perf_counter() - start
    return Reply(provider=provider, model=model, text=text.strip(), seconds=seconds,
                 input_tokens=in_tok, output_tokens=out_tok,
                 cost_usd=estimate_cost(provider, in_tok, out_tok), demo=demo)

## 3. Showdown ke functions
`run_showdown` har provider ko prompt bhejta hai (ek fail ho to baaki chalte hain), `show_results` jawab aur muqabla dikhata hai.

In [4]:
DEMO_ANSWERS = {
    "openai": "Python ek aasan programming language hai jo AI, web aur automation mein bohat use hoti hai.",
    "claude": "Python ek parhne mein saaf aur seekhne mein aasan language hai. Isi liye data science mein mashhoor hai.",
    "gemini": "Python ek high-level language hai. Is ki library (pandas, numpy) data ke kaam ko bohat aasan bana deti hai.",
    "groq": "Python: simple syntax, bari community, AI ke liye best. (Groq par llama model bohat tez jawab deta hai.)",
}


def run_showdown(prompt, providers, demo, temperature, max_tokens):
    results = []
    for name in providers:
        print(f"  -> {name} ko bhej raha hoon ...")
        try:
            reply = ask(name, prompt, temperature=temperature, max_tokens=max_tokens,
                        demo=demo, mock_reply=DEMO_ANSWERS[name])
            results.append(reply)
        except LLMError as e:
            # ek provider fail ho to baaki chalte rahen
            print(f"     [!] {e}")
    return results


def show_results(results):
    print()
    for r in results:
        tag = " (DEMO)" if r.demo else ""
        print("=" * 70)
        print(f"Provider : {r.provider}{tag}   Model: {r.model}")
        print(f"Time     : {r.seconds:.2f} sec")
        print(f"Tokens   : in={r.input_tokens}  out={r.output_tokens}  total={r.total_tokens}")
        print(f"Cost     : ~${r.cost_usd:.6f}  (andaza)")
        print("-" * 70)
        print(r.text)
    print("=" * 70)

    if results:
        print("\nMUQABLA")
        print(f"{'Provider':<10}{'Time(s)':>9}{'Tokens':>9}{'Cost($)':>12}")
        for r in sorted(results, key=lambda x: x.seconds):      # tez wala upar
            print(f"{r.provider:<10}{r.seconds:>9.2f}{r.total_tokens:>9}{r.cost_usd:>12.6f}")
        fastest = min(results, key=lambda x: x.seconds)
        cheapest = min(results, key=lambda x: x.cost_usd)
        print(f"\nSab se tez   : {fastest.provider}")
        print(f"Sab se sasta : {cheapest.provider}")

## 4. Chalao
Sirf `PROMPT` badlo. `CHOOSE = None` ka matlab: jin ki key `.env` mein hai.

In [5]:
PROMPT = "Python kya hai?"
CHOOSE = None               # ya ["groq", "openai"]
FORCE_DEMO = False

chosen = CHOOSE or available_providers()
demo = FORCE_DEMO
if not chosen:
    print("[!] .env mein koi API key nahi mili, DEMO mode chal raha hai.\n")
    demo = True
if demo:
    chosen = list(PROVIDERS)

results = run_showdown(PROMPT, chosen, demo, temperature=0.7, max_tokens=300)
show_results(results)

[!] .env mein koi API key nahi mili, DEMO mode chal raha hai.

  -> openai ko bhej raha hoon ...
  -> claude ko bhej raha hoon ...
  -> gemini ko bhej raha hoon ...
  -> groq ko bhej raha hoon ...

Provider : openai (DEMO)   Model: gpt-4o-mini
Time     : 0.12 sec
Tokens   : in=3  out=22  total=25
Cost     : ~$0.000014  (andaza)
----------------------------------------------------------------------
Python ek aasan programming language hai jo AI, web aur automation mein bohat use hoti hai.
Provider : claude (DEMO)   Model: claude-3-5-haiku-latest
Time     : 0.18 sec
Tokens   : in=3  out=26  total=29
Cost     : ~$0.000106  (andaza)
----------------------------------------------------------------------
Python ek parhne mein saaf aur seekhne mein aasan language hai. Isi liye data science mein mashhoor hai.
Provider : gemini (DEMO)   Model: gemini-1.5-flash
Time     : 0.09 sec
Tokens   : in=3  out=26  total=29
Cost     : ~$0.000008  (andaza)
--------------------------------------------------

## Kya seekha
- Har provider ka SDK alag hai, lekin request ke hisse wahi hain: `model`, `messages`, `max_tokens`, `temperature`.
- Claude mein `system` alag parameter hota hai aur `max_tokens` zaroori hai.
- Kharcha = input tokens x rate + output tokens x rate. Groq tez aur sasta, Claude / GPT zyada mehnge.
- Key hamesha `.env` ya Colab Secrets mein, code mein kabhi nahi.